# Skenario 01 — Optimasi PSO untuk Random Forest dan SVR

Notebook ini memakai data, target, fitur, pembagian acak, dan ruang pencarian yang sama dengan notebook ABC. Perbedaannya hanya metode optimasi: posisi, kecepatan, personal best, dan global best mengikuti Particle Swarm Optimization (PSO).

In [ ]:
from pathlib import Path
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

SEED = 42
np.random.seed(SEED)

## Konfigurasi dan persiapan data

In [ ]:
SENSOR_COLS = [
    "Exhaust Gas Temperature LF (Left Front)",
    "Exhaust Gas Temperature LR (Left Rear)",
    "Exhaust Gas Temperature RF (Right Front)",
    "Exhaust Gas Temperature RR (Right Rear)",
    "Blowby Pressure (Kpa)",
    "Boost Pressure (Kpa)",
    "Engine Oil Temp (°C)",
    "Coolant Temp (°C)",
]
LIFE_COL = "Unit Lifetime (Hour)"
STATUS_COL = "Status"
TARGET_COL = "sisa jam"
FEATURE_COLS = [*SENSOR_COLS, LIFE_COL]

cfg = {
    "data_file": "datas/Data_PT.Amanah_Critical_Break.xlsx",
    "max_rul": 2550.0,
    "test_size": 0.20,
    "cv_folds": 5,
    "swarm_size": 20,
    "iterations": 30,
    "inertia": 0.7,
    "cognitive": 1.5,
    "social": 1.5,
}
RF_KEYS = ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features"]
RF_BOUNDS = np.array([[50, 300], [2, 30], [2, 20], [1, 10], [0.1, 1.0]], dtype=float)
SVR_KEYS = ["C", "epsilon", "gamma"]
SVR_BOUNDS = np.array([[0.1, 1000], [0.001, 50], [0.0001, 1]], dtype=float)

def find_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / cfg["data_file"]).exists():
            return path
    raise FileNotFoundError(f"Dataset tidak ditemukan: {cfg['data_file']}")

def prepare_data(root):
    raw = pd.read_excel(root / cfg["data_file"])
    data = raw[[*FEATURE_COLS, STATUS_COL]].copy()
    for col in FEATURE_COLS:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    data[STATUS_COL] = data[STATUS_COL].astype("string").str.strip().str.casefold()
    break_hour = data.loc[data[STATUS_COL].eq("break"), LIFE_COL].max()
    if pd.isna(break_hour):
        raise ValueError("Baris berstatus Break diperlukan untuk membentuk target sisa jam")
    data[TARGET_COL] = (break_hour - data[LIFE_COL]).clip(lower=0, upper=cfg["max_rul"])
    data = data.loc[data[STATUS_COL].eq("critical")].dropna(subset=[*FEATURE_COLS, TARGET_COL]).copy()
    if data.empty:
        raise ValueError("Data Critical yang valid tidak ditemukan")
    return data, float(break_hour)

root = find_root()
out_dir = Path.cwd() / "artifacts" if Path.cwd().name == "01_forecast_sisa_jam_critical" else root / "newcode/scenarios/01_forecast_sisa_jam_critical/artifacts"
out_dir.mkdir(parents=True, exist_ok=True)
data, break_hour = prepare_data(root)
train_df, test_df = train_test_split(data, test_size=cfg["test_size"], shuffle=True, random_state=SEED)
X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]
cv = KFold(n_splits=cfg["cv_folds"], shuffle=True, random_state=SEED)
print({"baris_critical": len(data), "jam_break": break_hour, "fitur": FEATURE_COLS})

## Model, fungsi objektif, dan Particle Swarm Optimization

In [ ]:
def decode_rf(pos):
    return {
        "n_estimators": int(round(pos[0])),
        "max_depth": int(round(pos[1])),
        "min_samples_split": int(round(pos[2])),
        "min_samples_leaf": int(round(pos[3])),
        "max_features": float(pos[4]),
    }

def decode_svr(pos):
    return {"C": float(pos[0]), "epsilon": float(pos[1]), "gamma": float(pos[2]), "kernel": "rbf"}

def make_model(kind, params):
    if kind == "rf":
        return RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
    return Pipeline([("scale", StandardScaler()), ("svr", SVR(**params))])

def make_objective(kind):
    decode = decode_rf if kind == "rf" else decode_svr
    def objective(pos):
        model = make_model(kind, decode(pos))
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_mean_squared_error", n_jobs=1)
        return float(-scores.mean())
    return objective

def run_pso(objective, bounds, swarm_size=20, iterations=30, inertia=0.7, cognitive=1.5, social=1.5, seed=42):
    rng = np.random.default_rng(seed)
    low, high = bounds[:, 0], bounds[:, 1]
    span = high - low
    pos = rng.uniform(low, high, size=(swarm_size, len(bounds)))
    vel = rng.uniform(-0.1 * span, 0.1 * span, size=pos.shape)
    fit = np.array([objective(item) for item in pos])
    pbest_pos, pbest_fit = pos.copy(), fit.copy()
    best_idx = int(np.argmin(pbest_fit))
    gbest_pos, gbest_fit = pbest_pos[best_idx].copy(), float(pbest_fit[best_idx])
    history = [gbest_fit]
    for _ in range(iterations):
        r1 = rng.random(pos.shape)
        r2 = rng.random(pos.shape)
        vel = inertia * vel + cognitive * r1 * (pbest_pos - pos) + social * r2 * (gbest_pos - pos)
        vel = np.clip(vel, -span, span)
        pos = np.clip(pos + vel, low, high)
        fit = np.array([objective(item) for item in pos])
        improved = fit < pbest_fit
        pbest_pos[improved] = pos[improved]
        pbest_fit[improved] = fit[improved]
        best_idx = int(np.argmin(pbest_fit))
        if pbest_fit[best_idx] < gbest_fit:
            gbest_pos, gbest_fit = pbest_pos[best_idx].copy(), float(pbest_fit[best_idx])
        history.append(gbest_fit)
    return gbest_pos, gbest_fit, history

## Jalankan optimasi dan evaluasi

Cell berikut menjalankan optimasi penuh untuk dua model. Waktu proses bergantung pada perangkat.

In [ ]:
def evaluate(kind, params, cv_mse, history, elapsed):
    model = make_model(kind, params)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = {
        "optimizer": "PSO",
        "model": "Random Forest" if kind == "rf" else "SVR",
        "target": TARGET_COL,
        "features": FEATURE_COLS,
        "break_hour": break_hour,
        "best_params": params,
        "cv_mse": float(cv_mse),
        "cv_rmse": float(np.sqrt(cv_mse)),
        "test_mse": float(mean_squared_error(y_test, pred)),
        "test_rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
        "test_mae": float(mean_absolute_error(y_test, pred)),
        "test_r2": float(r2_score(y_test, pred)),
        "elapsed_seconds": float(elapsed),
        "seed": SEED,
    }
    model_dir = out_dir / f"pso_{kind}"
    model_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_dir / "model.joblib")
    (model_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    pd.DataFrame({"actual_sisa_jam": y_test, "predicted_sisa_jam": pred}).to_csv(model_dir / "test_predictions.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(history)
    axes[0].set(title=f"Konvergensi PSO {metrics['model']}", xlabel="Iterasi", ylabel="MSE CV terbaik")
    axes[1].scatter(y_test, pred, alpha=0.7)
    edge = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    axes[1].plot(edge, edge)
    axes[1].set(title=f"Aktual vs Prediksi {metrics['model']}", xlabel="Aktual sisa jam", ylabel="Prediksi sisa jam")
    fig.tight_layout()
    fig.savefig(model_dir / "evaluation.png", dpi=160, bbox_inches="tight")
    plt.show()
    return metrics

results = {}
for idx, (kind, bounds, decode) in enumerate([("rf", RF_BOUNDS, decode_rf), ("svr", SVR_BOUNDS, decode_svr)]):
    start = time.perf_counter()
    pos, score, history = run_pso(make_objective(kind), bounds, cfg["swarm_size"], cfg["iterations"], cfg["inertia"], cfg["cognitive"], cfg["social"], SEED + idx)
    results[kind] = evaluate(kind, decode(pos), score, history, time.perf_counter() - start)
pd.DataFrame(results).T[["cv_rmse", "test_rmse", "test_mae", "test_r2"]]